In [2]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer
import sentencepiece

In [33]:
eg=pd.read_parquet("Egy.parquet")
sm1=pd.read_json("sumarabic_test.jsonl", lines=True)
sm2=pd.read_json("sumarabic_train.jsonl", lines=True)
sm3=pd.read_json("sumarabic_valid.jsonl", lines=True)
smds=pd.read_csv("summarizedd.csv")

In [35]:
AraSum = pd.read_csv("ArabicSum.csv" , sep = "\t" , engine = "python" , header = None , names = ["index" , "text" , "summarized"])
AraSum = AraSum.drop("index" , axis = 1)

In [36]:
sm_train=sm2[["text","headline"]]
sm_train=sm_train.rename(columns={"headline":"summarized"})

In [6]:
sm_test=sm1[["text","headline"]]
sm_test=sm_test.rename(columns={"headline":"summarized"})

In [7]:
sm_valid=sm3[["text","headline"]]
sm_valid=sm_valid.rename(columns={"headline":"summarized"})

In [8]:
eg.head()

,text,summarized_text,source_topics
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...,تأثير الأعباء الاقتصادية على قرارات الإنجاب
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...,تأثير الأعباء الاقتصادية على قرارات الإنجاب


In [9]:
eg=eg[["text","summarized_text"]].rename(columns={"summarized_text":"summarized"})

In [10]:
smds.head()

,text,type,Processed Text,summarizer
0,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...,culture,اشرف رئيس الجمهوريه الباجي قايد السبسي اليوم ب...,\nأشرف رئيس الجمهورية الباجي قايد السبسي اليوم...
1,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون...",culture,تحصل كتاب المصحف وقراءاته الفه باحثون تونسيون ...,"\nتحصل كتاب ""المصحف وقراءاته"" الذي ألفه باحثون..."
2,\nاحتضن جناح تونس في القرية الدولية للأفلام بم...,culture,احتضن جناح تونس القريه الدوليه للافلام بمدينه ...,تونس حاضرة من جهة أخرى ستكون تونس حاضرة في قائ...
3,\nشهدت برلين أمس الجمعة افتتاح مسجد فريد من نو...,culture,شهدت برلين الجمعه افتتاح مسجد فريد نوعه الاقل ...,واستأجرت صاحبة المشروع المحامية والكاتبة سيران...
4,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...,culture,نعت وزاره المنشد عز بن محمود انتقل جوار يوم تن...,\nنعت وزارة الشّؤون الثّقافيّة المنشد الصّوفي ...


In [11]:
smds=smds[["text","summarizer"]].rename(columns={"summarizer":"summarized"})

In [12]:
AraSum.head()

,text,target
0,"حقق حزب ""البديل من أجل ألمانيا"" اليميني الشعبو...",تشير آخر استطلاعات الرأي الألمانية إلى تقدم حز...
1,بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ...,بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل ...
2,قال مسؤولون إن شخصا أصيب إصابة بالغة بعد تعرضه...,أعلن حاكم كارولاينا الشمالية حالة الطورائ في م...
3,أعلن مسؤول في البنتاغون أن جنودا أمريكيين في ش...,قالت وزارة الدفاع الأمريكية (البنتاغون) إن داع...
4,اجتمعت المجموعة الدولية لدعم سوريا على هامش اج...,فشلت الولايات المتحدة وروسيا في الاتفاق على كي...


In [14]:
concat_df=pd.concat([eg,smds,AraSum],ignore_index=True)

In [15]:
concat_df.head()

,text,summarized
0,كتير من الشباب دلوقتي بيفكروا ألف مرة قبل ما ي...,غلاء المعيشة بيخلي الشباب يقللوا عدد الأطفال ب...
1,الظروف الاقتصادية الصعبة خلت كتير من الشباب يأ...,الوضع الاقتصادي المتردي بيأجل قرارات الجواز وا...
2,موجات التضخم المتتالية أثرت بشكل مباشر على قدر...,التضخم بيخلي الأسر تعيد حساباتها في قرارات الإ...
3,الأسر المصرية بقت بتميل أكتر لإنها تركز على جو...,الأعباء الاقتصادية بتخلي الأسر تركز على جودة ح...
4,في ظل التحديات الاقتصادية اللي بتواجه الأسر، ب...,الحكومات ممكن تحتاج تقدم دعم وحوافز للأسر عشان...


In [16]:
from sklearn.model_selection import train_test_split
concat_df_train,concat_df_test = train_test_split( concat_df,test_size=0.15, random_state=42
)
concat_df_train2,concat_df_validat = train_test_split( concat_df,test_size=0.15, random_state=42
)

In [17]:
concat_df_train2

,text,summarized
10453,\nقال الناشط الحقوقي والمهتم بالشأن الليبي مصط...,\nقال الناشط الحقوقي والمهتم بالشأن الليبي مصط...
40384,تقول الحكومة المصرية إن مشكلة الكلاب الضالة با...,"""ضد تسميم الكلاب"" و""ضد تصدير الكلاب"" و""لا لانت..."
45160,حكمت محكمة ألمانية اليوم الجمعة (10 شباط/ فبرا...,بعد مرور عام تقريباً على الهجوم الذي استهدف جن...
8528,\nنشرت بلدية تونس العاصمة صورا لتسماح بحديقة ا...,"وأكدت المكلفة بالاعلام ببلدية تونس للجوهرة ""اف..."
38154,أظهر استطلاع للرأي اليوم الخميس (13 حزيران/ يو...,"في أحدث استطلاع رأي بتكليف من منظمة ""السلام ال..."
...,...,...
54343,شهدت الساعات القليلة الماضية توتراً حاداً بين ...,اتفق الرئيسان التركي والأمريكي على إقامة منطقة...
38158,"""أزمة في بايرن""، خبر معتاد قادم من عرين النادي...",أزمة في بايرن أم لا؟ بعد تعادلين وخسارة بدأ ال...
860,قيادة الاستدامة في إدارة المخلفات بتبدأ بوضع ا...,وضع استراتيجية مستدامة لإدارة المخلفات بيركز ع...
15795,وجه البابا فرنسيس اليوم الأحد (25 مايو/ أيار 2...,دعا البابا فرنسيس الرئيسين الفلسطيني والإسرائي...


In [18]:
final_df_train=pd.concat([sm_train,concat_df_train2],ignore_index=True)
final_df_test=pd.concat([sm_test,concat_df_test],ignore_index=True)
final_df_valid=pd.concat([sm_valid,concat_df_validat],ignore_index=True)

In [19]:
final_df_train.head()

,text,summarized
0,اختتمت مساء أول من أمس نهائيات بطولة الإمارات ...,المصري فؤاد الطاهر بطل للشطرنج الديناميكي
1,مينيابوليس (الولايات المتحدة) 3-2-2008 (ا ف ب)...,"حملة انتخابية محمومة قبل ""الثلاثاء الكبير"""
2,أفاد مصدر في شرطة أبوظبي بأن نحو 700 طفل يتواف...,55% من نزلاء «الأحداث» مواطنون
3,قال استشاري أمراض الكلى في مدينة الشيخ خليفة ا...,مركز متخصص في الكلى بمستشفى خليفة
4,بدأت القيادة العامة لشرطة أبوظبي التنسيق لإنشا...,تأهيل ضحايا حوادث المرور في أبوظبي


In [20]:
import re

def clean_arabic_text(text):
    if not isinstance(text, str):
        return ""

    #Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)

    #Remove emails
    text = re.sub(r'\S+@\S+\.\S+', ' ', text)

    #Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)

    #Remove script/style blocks
    text = re.sub(r'<script.*?>.*?</script>', ' ', text, flags=re.IGNORECASE | re.DOTALL)

    #Remove CSS styles
    text = re.sub(r'<style.*?>.*?</style>', ' ', text, flags=re.IGNORECASE | re.DOTALL)

    #Remove mentions
    text = re.sub(r'@\w+', ' ', text)

    #Remove hashtag symbol but keep word
    text = re.sub(r'#(\w+)', r'\1', text)

    #Remove tatweel
    text = re.sub(r'ـ+', '', text)

    #Remove English letters
    text = re.sub(r'[a-zA-Z]', '', text)

    #Remove parentheses and slashes
    text = re.sub(r'[()/]', ' ', text)

    #Remove single isolated Arabic letters
    text = re.sub(r'\s[ء-ي]\s', ' ', text)

    #Keep Arabic + punctuation only
    text = re.sub(r'[^\u0600-\u06FF.,!?؟،:؛()\-\n\s]', ' ', text)

    #Normalize punctuation
    text = text.replace('…', '.')
    text = text.replace('٪', '%')
    text = text.replace('―', '-')

    #Normalize spaces
    text = re.sub(r'[ \t]+', ' ', text)

    #Normalize newlines
    text = re.sub(r'\n+', '\n', text)

    return text.strip()

In [21]:
final_df_train["clean_text"] =final_df_train["text"].apply(clean_arabic_text)
final_df_train["clean_summary"]= final_df_train["summarized"].apply(clean_arabic_text)

final_df_test["clean_text"] = final_df_test["text"].apply(clean_arabic_text)
final_df_test["clean_summary"]= final_df_test["summarized"].apply(clean_arabic_text)

final_df_valid["clean_text"] = final_df_valid["text"].apply(clean_arabic_text)
final_df_valid["clean_summary"]= final_df_valid["summarized"].apply(clean_arabic_text)

In [22]:
pip install transformers sentencepiece

In [23]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("UBC-NLP/AraT5-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [24]:
def tkn(data):
    inputs = tokenizer(
        "summarize: " + data["text"],
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    targets = tokenizer(
        data["summarized"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )

    labels = [
        (token if token != tokenizer.pad_token_id else -100)
        for token in targets["input_ids"]
    ]

    inputs["labels"] = labels
    return inputs

In [25]:
df_final_train_tokenized = final_df_train.apply(tkn, axis=1)

In [26]:
df_final_train_tokenized

,0
0,"[input_ids, attention_mask, labels]"
1,"[input_ids, attention_mask, labels]"
2,"[input_ids, attention_mask, labels]"
3,"[input_ids, attention_mask, labels]"
4,"[input_ids, attention_mask, labels]"
...,...
54769,"[input_ids, attention_mask, labels]"
54770,"[input_ids, attention_mask, labels]"
54771,"[input_ids, attention_mask, labels]"
54772,"[input_ids, attention_mask, labels]"


In [27]:
sample = df_final_train_tokenized.iloc[0]

print(sample["input_ids"][:20])
print(sample["labels"][:20])

print(tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
print(tokenizer.decode(
    [t for t in sample["labels"] if t != -100],
    skip_special_tokens=True
))

[35880, 4633, 16410, 33, 33261, 527, 416, 6, 397, 13239, 1262, 1071, 97891, 468, 6309, 31, 20, 15068, 53, 40977]
[920, 5451, 16820, 1992, 97891, 468, 6309, 31, 20, 1, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100]
summarize: اختتمت مساء أول من أمس نهائيات بطولة الإمارات للشطرنج الديناميكي المفتوحة التي نظمها اتحاد الإمارات للشطرنج بمقر نادي دبي للشطرنج والثقافة حيث توج الأستاذ الدولي المصري فؤاد الطاهر بطلا للمسابقة برصيد 6 نقاط من 7 جولات.
المصري فؤاد الطاهر بطل للشطرنج الديناميكي


In [28]:
sample = df_final_train_tokenized.iloc[0]

In [29]:
sample

{'input_ids': [35880, 4633, 16410, 33, 33261, 527, 416, 6, 397, 13239, 1262, 1071, 97891, 468, 6309, 31, 20, 15068, 53, 40977, 1603, 1071, 97891, 7220, 739, 1106, 97891, 11625, 178, 16724, 3643, 675, 920, 5451, 16820, 14756, 58487, 5050, 488, 1904, 6, 579, 17012, 4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0